In [1]:
# ==========================================
# CELL 1: Import Libraries and Setup
# ==========================================
from openai import OpenAI
import pandas as pd
import logging
from typing import Dict, List, Tuple
from tqdm import tqdm

# Setup logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

print("✓ Libraries imported successfully")


✓ Libraries imported successfully


In [ ]:
# ==========================================
# CELL 2: Initialize OpenTyphoon Client
# ==========================================
client = OpenAI(
    api_key="",
    base_url="https://api.opentyphoon.ai/v1"
)

print("✓ OpenTyphoon client initialized")

✓ OpenTyphoon client initialized


In [12]:
# ==========================================
# CELL 3: Define System Prompt
# ==========================================
SYSTEM_PROMPT = """You are an expert Thai legal classifier. Your task is to determine if a question is related to Thai Civil and Commercial Code (ประมวลกฎหมายแพ่งและพาณิชย์) or not.

=== CORE DEFINITIONS ===

Civil law (กฎหมายแพ่ง) refers to laws concerning the rights and duties of individuals, such as matters of personal status, property, obligations, legal acts, family, and inheritance.
A civil wrong is an act that causes harm to a specific individual rather than the public at large.
Commercial law (กฎหมายพาณิชย์) deals with rights and duties in economic and trade activities, regulating business relations between individuals, such as the establishment of partnerships or companies, carriage of goods, and negotiable instruments (like cheques).

Civil and Commercial Code (CCC) covers PRIVATE LAW between individuals:

CIVIL LAW (กฎหมายแพ่ง):
- Personal status and capacity (บุคคล, นิติบุคคล)
- Property rights (ทรัพย์สิน, กรรมสิทธิ์, ทรัพยสิทธิ)
- Obligations and contracts (นิติกรรม, สัญญา, หนี้)
- Family law (สมรส, บุตร, อุปการะ)
- Inheritance (มรดก, พินัยกรรม)
- Torts (ละเมิด)


COMMERCIAL LAW (กฎหมายพาณิชย์):
- Business entities (ห้างหุ้นส่วน, บริษัท)
- Commercial contracts (ซื้อขาย, เช่าซื้อ, ตัวแทน, นายหน้า)
- Secured transactions (จำนอง, จำนำ, ประกัน)
- Negotiable instruments (ตั๋วเงิน, เช็ค)
- Carriage of goods (ขนส่ง)

=== EXCLUSIONS (NOT CCC) ===

Criminal Law (กฎหมายอาญา):
- Crimes, penalties, imprisonment
- Murder, theft, fraud, assault
- Criminal procedure

Public/Administrative Law (กฎหมายมหาชน):
- Government operations
- Public officials
- Administrative procedure
- Constitutional law

Special Laws (กฎหมายพิเศษ):
- Tax law (ภาษีอากร)
- Labor law (แรงงาน, ประกันสังคม)
- Securities/Capital market (หลักทรัพย์, ตลาดหลักทรัพย์)
- Banking regulations (ธนาคาร, สถาบันการเงิน)
- Insurance regulations (ประกันภัย - ต่างจากการประกันในแพ่งและพาณิชย์)
- Intellectual property (ลิขสิทธิ์, เครื่องหมายการค้า, สิทธิบัตร)
- Consumer protection (คุ้มครองผู้บริโภค)
- Land Code (ประมวลกฎหมายที่ดิน - ต่างจากทรัพยสิทธิในแพ่งและพาณิชย์)
- Bankruptcy (ล้มละลาย - แม้เกี่ยวข้องกับหนี้)

=== DECISION RULES ===

1. If the question asks about RELATIONSHIPS, TRANSACTIONS, or DISPUTES between PRIVATE parties : likely YES
2. If the question asks about CRIMES, PENALTIES, or STATE PROSECUTION : NO
3. If the question asks about TAX, LICENSING, or GOVERNMENT APPROVAL : NO
4. If the question involves SPECIAL STATUTORY REGIMES (securities, banking, IP) : NO
5. For MIXED questions, classify based on PRIMARY focus

=== EXAMPLES ===

Example 1:
Question: การทำสัญญาเช่าบ้านต้องทำเป็นหนังสือหรือไม่
Answer: YES
Reason: เกี่ยวกับสัญญาเช่าซึ่งเป็นสัญญาแต่ละชนิดในหมวดหนี้ ประมวลกฎหมายแพ่งและพาณิชย์

Example 2:
Question: การจดทะเบียนบริษัทจำกัดต้องมีทุนจดทะเบียนขั้นต่ำเท่าไหร่
Answer: YES
Reason: เกี่ยวกับบริษัทจำกัดในหมวดพาณิชย์ ประมวลกฎหมายแพ่งและพาณิชย์

Example 3:
Question: การฆ่าคนตายมีโทษอย่างไร
Answer: NO
Reason: เกี่ยวกับความผิดอาญา อยู่ในประมวลกฎหมายอาญา

Example 4:
Question: ในกรณีที่บุคคลหนึ่งได้ออกใบรับหลายๆฉบับอันเป็นการแบ่งแยกมูลค่า เพื่อหลีกเลี่ยงการเสียอากร มีความผิดหรือไม่
Answer: NO
Reason: เกี่ยวกับการเลี่ยงภาษีอากรแสตมป์ ซึ่งเป็นกฎหมายภาษีอากร ไม่ใช่แพ่งและพาณิชย์

Example 5:
Question: การเลิกจ้างพนักงานต้องแจ้งล่วงหน้ากี่วัน
Answer: NO
Reason: เกี่ยวกับกฎหมายคุ้มครองแรงงาน ไม่ใช่สัญญาจ้างแรงงานทั่วไปในแพ่งและพาณิชย์

Example 6:
Question: นายจ้างไม่จ่ายค่าจ้างตามสัญญาจ้างทำของ ลูกจ้างฟ้องเรียกค่าเสียหายได้หรือไม่
Answer: YES
Reason: เกี่ยวกับสัญญาจ้างทำของและการผิดสัญญา อยู่ในประมวลกฎหมายแพ่งและพาณิชย์

Example 7:
Question: การขอสิทธิบัตรการประดิษฐ์ต้องยื่นคำขอที่ไหน
Answer: NO
Reason: เกี่ยวกับทรัพย์สินทางปัญญา (สิทธิบัตร) อยู่ในพระราชบัญญัติสิทธิบัตร

Example 8:
Question: เจ้าของที่ดินสามารถขอโฉนดที่ดินได้อย่างไร
Answer: NO
Reason: เกี่ยวกับการออกโฉนดที่ดิน อยู่ในประมวลกฎหมายที่ดิน แม้จะเกี่ยวข้องกับทรัพย์สิน

Example 9:
Question: ผู้เยาว์อายุ 16 ปีทำนิติกรรมได้หรือไม่
Answer: YES
Reason: เกี่ยวกับความสามารถของบุคคลและการทำนิติกรรม อยู่ในหมวดบุคคล ประมวลกฎหมายแพ่งและพาณิชย์

Example 10:
Question: บริษัทที่ล้มละลายต้องดำเนินการอย่างไร
Answer: NO
Reason: เกี่ยวกับล้มละลาย อยู่ในพระราชบัญญัติล้มละลาย ไม่ใช่แพ่งและพาณิชย์

Example 11:
Question: การทำพินัยกรรมต้องมีพยานกี่คน
Answer: YES
Reason: เกี่ยวกับพินัยกรรมในหมวดมรดก ประมวลกฎหมายแพ่งและพาณิชย์

Example 12:
Question: ธนาคารแห่งประเทศไทยมีอำนาจกำกับดูแลธนาคารพาณิชย์อย่างไร
Answer: NO
Reason: เกี่ยวกับการกำกับดูแลสถาบันการเงิน อยู่ในกฎหมายธนาคารพาณิชย์


=== CLASSIFICATION TASK ===

Analyze the question step by step:
1. Identify the main legal topic
2. Determine if it involves private relations (CCC) or public law/special law
3. Check against exclusion list
Respond with only "YES" if the question is related to Civil and Commercial Code, or "NO" if it's related to other types of law or non-legal topics.
"""

print("✓ System prompt defined")

✓ System prompt defined


In [4]:
# ==========================================
# CELL 4: Define Classification Function
# ==========================================
def classify_legal_question(user_input: str) -> Dict:
    """
    Classify if a question is related to Civil and Commercial Code

    Args:
        user_input: The question to classify

    Returns:
        Dictionary with classification results
    """
    try:
        # Create the full prompt
        user_message = f"Question: {user_input}\n\nAnswer:"

        # Make API call
        response = client.chat.completions.create(
            model="typhoon-v2.5-30b-a3b-instruct",
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": user_message}
            ],
            max_tokens=10,  # Only need YES/NO
            temperature=0.1
        )

        # Extract response
        response_text = response.choices[0].message.content.strip().upper()

        # Determine classification
        if "YES" in response_text:
            return {
                "passed": True,
                "reason": "Input is related to Civil and Commercial Code",
                "decision": "allowed",
                "raw_response": response_text
            }
        else:
            return {
                "passed": False,
                "reason": "Input is not related to Civil and Commercial Code",
                "decision": "not allowed",
                "raw_response": response_text
            }

    except Exception as e:
        logger.error(f"Error in classification: {str(e)}")
        return {
            "passed": False,
            "reason": f"Error: {str(e)}",
            "decision": "error",
            "raw_response": ""
        }

print("✓ Classification function defined")

✓ Classification function defined


In [5]:
# ==========================================
# CELL 5: Test Single Classification
# ==========================================
print("Testing single classification:\n")

# Test with non-legal question
test_question_1 = "ขอสูตรไก่ย่าง"
result_1 = classify_legal_question(test_question_1)
print(f"Question 1: {test_question_1}")
print(f"Result: {result_1}\n")

# Test with legal question
test_question_2 = "การทำสัญญาเช่าบ้านต้องมีหลักฐานเป็นลายลักษณ์อักษรหรือไม่"
result_2 = classify_legal_question(test_question_2)
print(f"Question 2: {test_question_2}")
print(f"Result: {result_2}")

Testing single classification:

Question 1: ขอสูตรไก่ย่าง
Result: {'passed': False, 'reason': 'Input is not related to Civil and Commercial Code', 'decision': 'not allowed', 'raw_response': 'NO'}

Question 2: การทำสัญญาเช่าบ้านต้องมีหลักฐานเป็นลายลักษณ์อักษรหรือไม่
Result: {'passed': True, 'reason': 'Input is related to Civil and Commercial Code', 'decision': 'allowed', 'raw_response': 'YES'}


In [6]:
# ==========================================
# CELL 6: Define Data Loading Function (Updated)
# ==========================================
import ast  # Import the ast module
import json  # Keep json import in case needed elsewhere

def load_and_transform_data(csv_path: str) -> pd.DataFrame:
    """Load and transform the CCL dataset for testing"""
    try:
        df = pd.read_csv(csv_path)
        logger.info(f"Loaded {len(df)} records from {csv_path}")

        # Transform the data for testing
        test_data = []

        for idx, row in df.iterrows():
            question = row['question']
            relevant_laws_str = row['relevant_laws']

            relevant_laws = []
            is_civil_commercial = False
            law_names = []

            # Attempt to parse the string as a Python literal (handles single quotes)
            try:
                parsed_laws = ast.literal_eval(relevant_laws_str)
                if isinstance(parsed_laws, list):
                    relevant_laws = parsed_laws
                else:
                    logger.warning(f"Row {idx}: Parsed data is not a list: {relevant_laws_str}")

            except (ValueError, SyntaxError) as e:  # Catch ValueError and SyntaxError for ast.literal_eval
                logger.warning(f"Row {idx}: Failed to parse literal: {relevant_laws_str} - {e}")
                continue  # Skip this row if parsing fails

            # Process the parsed relevant_laws list
            for law in relevant_laws:
                if isinstance(law, dict):
                    law_name = law.get('law_name', '')
                    law_names.append(law_name)

                    # Check if it contains Civil and Commercial Code keywords
                    if any(keyword in law_name for keyword in [
                        "ประมวลกฎหมายแพ่งและพาณิชย์",
                        "แพ่งและพาณิชย์",
                        "Civil and Commercial Code"
                    ]):
                        is_civil_commercial = True
                        break
                else:
                    logger.warning(f"Row {idx}: Item in relevant_laws is not a dictionary: {law}")

            expected_decision = "allowed" if is_civil_commercial else "not allowed"

            test_data.append({
                'question': question,
                'relevant_laws': relevant_laws,
                'law_names': law_names,
                'is_civil_commercial': is_civil_commercial,
                'expected_decision': expected_decision,
                'original_answer': row.get('answer', ''),
            })

        test_df = pd.DataFrame(test_data)
        logger.info(f"Transformed {len(test_df)} records successfully")

        return test_df

    except Exception as e:
        logger.error(f"Error loading data: {e}")
        raise

print("✓ Data loading function defined (with relevant_laws parsing)")

✓ Data loading function defined (with relevant_laws parsing)


In [8]:
# ==========================================
# CELL 7: Load and Preview Test Data (Updated)
# ==========================================
# Load CSV - Update this path to your file location
csv_path = "/content/ccl.csv"

logger.info("Loading and transforming test data...")
test_df = load_and_transform_data(csv_path)

# Display data distribution
print(f"\n{'='*60}")
print("DATA DISTRIBUTION")
print(f"{'='*60}")
print(f"Total samples: {len(test_df)}")
print(f"Civil & Commercial: {sum(test_df['is_civil_commercial'])}")
print(f"Other laws: {sum(~test_df['is_civil_commercial'])}")
print(f"{'='*60}")

# Display column information
print(f"\nDataFrame columns: {list(test_df.columns)}")
print(f"DataFrame shape: {test_df.shape}")

# Preview first few samples with detailed information
print("\n" + "="*60)
print("FIRST 5 SAMPLES (DETAILED)")
print("="*60)

for i in range(min(5, len(test_df))):
    row = test_df.iloc[i]
    print(f"\n{'---'*20}")
    print(f"Sample {i+1}")
    print(f"{'---'*20}")
    print(f"Question: {row['question'][:150]}...")
    print(f"Expected Decision: {row['expected_decision']}")
    print(f"Is Civil & Commercial: {row['is_civil_commercial']}")
    print(f"Number of Relevant Laws: {len(row['relevant_laws'])}")

    # Show law names
    if row['law_names']:
        print(f"Law Names:")
        for j, law_name in enumerate(row['law_names'][:3], 1):  # Show first 3 laws
            print(f"  {j}. {law_name[:80]}...")
        if len(row['law_names']) > 3:
            print(f"  ... and {len(row['law_names']) - 3} more laws")
    else:
        print(f"Law Names: None")

# Show distribution by law type
print(f"\n{'='*60}")
print("DISTRIBUTION BY LAW TYPE")
print(f"{'='*60}")

# Count most common law names
all_law_names = []
for laws in test_df['law_names']:
    all_law_names.extend(laws)

if all_law_names:
    from collections import Counter
    law_counter = Counter(all_law_names)
    print(f"\nTop 10 Most Common Laws:")
    for law, count in law_counter.most_common(10):
        print(f"  {law[:70]}: {count} times")
else:
    print("No law names found in dataset")

# Show some examples of Civil & Commercial questions
print(f"\n{'='*60}")
print("CIVIL & COMMERCIAL EXAMPLES")
print(f"{'='*60}")
civil_examples = test_df[test_df['is_civil_commercial']].head(3)
for i, (idx, row) in enumerate(civil_examples.iterrows(), 1):
    print(f"\n{i}. {row['question'][:120]}...")
    print(f"   Laws: {', '.join([law[:50] for law in row['law_names'][:2]])}")

# Show some examples of Other law questions
print(f"\n{'='*60}")
print("OTHER LAW EXAMPLES")
print(f"{'='*60}")
other_examples = test_df[~test_df['is_civil_commercial']].head(3)
for i, (idx, row) in enumerate(other_examples.iterrows(), 1):
    print(f"\n{i}. {row['question'][:120]}...")
    if row['law_names']:
        print(f"   Laws: {', '.join([law[:50] for law in row['law_names'][:2]])}")
    else:
        print(f"   Laws: None")

print(f"\n{'='*60}")
print("✓ Data loaded and previewed successfully")
print(f"{'='*60}")


DATA DISTRIBUTION
Total samples: 3729
Civil & Commercial: 1617
Other laws: 2112

DataFrame columns: ['question', 'relevant_laws', 'law_names', 'is_civil_commercial', 'expected_decision', 'original_answer']
DataFrame shape: (3729, 6)

FIRST 5 SAMPLES (DETAILED)

------------------------------------------------------------
Sample 1
------------------------------------------------------------
Question: ถ้ามีคนประกอบกิจการในลักษณะเป็นศูนย์ซื้อขายสัญญาซื้อขายล่วงหน้าโดยไม่ได้รับใบอนุญาตต้องระวางโทษอย่างไร...
Expected Decision: not allowed
Is Civil & Commercial: False
Number of Relevant Laws: 1
Law Names:
  1. พระราชบัญญัติสัญญาซื้อขายล่วงหน้า พ.ศ. 2546...

------------------------------------------------------------
Sample 2
------------------------------------------------------------
Question: ถ้าผู้ิยู่ในปกครองได้ยินยอมในการกระทำของผู้ปกครองจะทำให้ผู้ปกครองหลุดพ้นจากความรับผิดหรือเปล่า...
Expected Decision: allowed
Is Civil & Commercial: True
Number of Relevant Laws: 1
Law Names:
  1. ปร

In [9]:
# ==========================================
# CELL 8: Define Evaluation Functions
# ==========================================
def run_evaluation(test_df: pd.DataFrame, sample_size: int = None) -> Tuple[Dict, pd.DataFrame]:
    """
    Run evaluation on test dataset

    Args:
        test_df: Test DataFrame
        sample_size: Number of samples to evaluate (None for all)

    Returns:
        Tuple of (metrics, results_df)
    """
    # Sample data if specified
    if sample_size and sample_size < len(test_df):
        eval_df = test_df.sample(n=sample_size, random_state=42).reset_index(drop=True)
    else:
        eval_df = test_df.copy()

    results = []

    # Evaluate each sample
    logger.info(f"Evaluating {len(eval_df)} samples...")
    for idx, row in tqdm(eval_df.iterrows(), total=len(eval_df), desc="Evaluating"):
        question = row['question']
        expected_decision = row['expected_decision']

        # Get classification
        result = classify_legal_question(question)

        # Store results
        results.append({
            'question': question,
            'expected_decision': expected_decision,
            'predicted_decision': result['decision'],
            'passed': result['passed'],
            'reason': result['reason'],
            'raw_response': result['raw_response'],
            'correct': (result['decision'] == expected_decision)
        })

    # Create results DataFrame
    results_df = pd.DataFrame(results)

    # Calculate metrics
    total = len(results_df)
    correct = results_df['correct'].sum()
    accuracy = correct / total if total > 0 else 0

    # Calculate confusion matrix
    true_positives = ((results_df['predicted_decision'] == 'allowed') &
                      (results_df['expected_decision'] == 'allowed')).sum()
    false_positives = ((results_df['predicted_decision'] == 'allowed') &
                       (results_df['expected_decision'] != 'allowed')).sum()
    false_negatives = ((results_df['predicted_decision'] != 'allowed') &
                       (results_df['expected_decision'] == 'allowed')).sum()
    true_negatives = ((results_df['predicted_decision'] != 'allowed') &
                      (results_df['expected_decision'] != 'allowed')).sum()

    # Calculate precision, recall, F1 for 'allowed' class
    precision_allowed = true_positives / (true_positives + false_positives) if (true_positives + false_positives) > 0 else 0
    recall_allowed = true_positives / (true_positives + false_negatives) if (true_positives + false_negatives) > 0 else 0
    f1_allowed = 2 * (precision_allowed * recall_allowed) / (precision_allowed + recall_allowed) if (precision_allowed + recall_allowed) > 0 else 0

    # Calculate precision, recall, F1 for 'not allowed' class
    precision_not_allowed = true_negatives / (true_negatives + false_negatives) if (true_negatives + false_negatives) > 0 else 0
    recall_not_allowed = true_negatives / (true_negatives + false_positives) if (true_negatives + false_positives) > 0 else 0
    f1_not_allowed = 2 * (precision_not_allowed * recall_not_allowed) / (precision_not_allowed + recall_not_allowed) if (precision_not_allowed + recall_not_allowed) > 0 else 0

    metrics = {
        'total': total,
        'correct': correct,
        'accuracy': accuracy,
        'true_positives': true_positives,
        'false_positives': false_positives,
        'false_negatives': false_negatives,
        'true_negatives': true_negatives,
        'precision_allowed': precision_allowed,
        'recall_allowed': recall_allowed,
        'f1_allowed': f1_allowed,
        'precision_not_allowed': precision_not_allowed,
        'recall_not_allowed': recall_not_allowed,
        'f1_not_allowed': f1_not_allowed
    }

    return metrics, results_df


def print_evaluation_results(metrics: Dict, results_df: pd.DataFrame, show_errors: int = 5):
    """
    Print evaluation results in a formatted way

    Args:
        metrics: Dictionary of evaluation metrics
        results_df: DataFrame with evaluation results
        show_errors: Number of errors to display (default: 5)
    """
    print("\n" + "="*60)
    print("EVALUATION RESULTS")
    print("="*60)
    print(f"Total samples: {metrics['total']}")
    print(f"Correct predictions: {metrics['correct']}")
    print(f"Accuracy: {metrics['accuracy']:.2%}")

    print(f"\n--- Confusion Matrix ---")
    print(f"True Positives (Allowed): {metrics['true_positives']}")
    print(f"False Positives (Allowed): {metrics['false_positives']}")
    print(f"False Negatives (Not Allowed): {metrics['false_negatives']}")
    print(f"True Negatives (Not Allowed): {metrics['true_negatives']}")

    print(f"\n--- Metrics for 'Allowed' Class ---")
    print(f"Precision: {metrics['precision_allowed']:.2%}")
    print(f"Recall: {metrics['recall_allowed']:.2%}")
    print(f"F1 Score: {metrics['f1_allowed']:.2%}")

    print(f"\n--- Metrics for 'Not Allowed' Class ---")
    print(f"Precision: {metrics['precision_not_allowed']:.2%}")
    print(f"Recall: {metrics['recall_not_allowed']:.2%}")
    print(f"F1 Score: {metrics['f1_not_allowed']:.2%}")

    print("="*60)

    # Show some incorrect predictions
    incorrect = results_df[~results_df['correct']]
    if len(incorrect) > 0:
        print(f"\nShowing first {min(show_errors, len(incorrect))} incorrect predictions:")
        for i, (idx, row) in enumerate(incorrect.head(show_errors).iterrows(), 1):
            print(f"\n{'---'*20}")
            print(f"Incorrect Prediction #{i}")
            print(f"{'---'*20}")
            print(f"Question: {row['question'][:150]}...")
            print(f"Expected: {row['expected_decision']}")
            print(f"Predicted: {row['predicted_decision']}")
            print(f"Raw Response: {row['raw_response']}")
    else:
        print("\n All predictions are correct!")

print("✓ Evaluation functions defined")

✓ Evaluation functions defined


In [10]:
# ==========================================
# CELL 9: Run Small Sample Evaluation (10 samples)
# ==========================================
print("Running evaluation on 10 samples...")
metrics_tiny, results_tiny = run_evaluation(test_df, sample_size=10)
print_evaluation_results(metrics_tiny, results_tiny)

Running evaluation on 10 samples...


Evaluating: 100%|██████████| 10/10 [00:09<00:00,  1.02it/s]


EVALUATION RESULTS
Total samples: 10
Correct predictions: 7
Accuracy: 70.00%

--- Confusion Matrix ---
True Positives (Allowed): 4
False Positives (Allowed): 1
False Negatives (Not Allowed): 2
True Negatives (Not Allowed): 3

--- Metrics for 'Allowed' Class ---
Precision: 80.00%
Recall: 66.67%
F1 Score: 72.73%

--- Metrics for 'Not Allowed' Class ---
Precision: 60.00%
Recall: 75.00%
F1 Score: 66.67%

Showing first 3 incorrect predictions:

------------------------------------------------------------
Incorrect Prediction #1
------------------------------------------------------------
Question: การตั้งชื่อสมาคมมีข้อกหนดหรือไม่...
Expected: allowed
Predicted: not allowed
Raw Response: NO

------------------------------------------------------------
Incorrect Prediction #2
------------------------------------------------------------
Question: หากได้รับความเสียหายจากการรับรองที่ไมไ่ด้มาตรฐานจะสามารถเรียกร้องค่าเสียหายได้หรือไม่...
Expected: not allowed
Predicted: allowed
Raw Response: YES


In [11]:
# ==========================================
# CELL 10: Run Medium Sample Evaluation (50 samples)
# ==========================================
print("Running evaluation on 50 samples...")
metrics_small, results_small = run_evaluation(test_df, sample_size=50)
print_evaluation_results(metrics_small, results_small)

Running evaluation on 50 samples...


Evaluating: 100%|██████████| 50/50 [00:34<00:00,  1.45it/s]


EVALUATION RESULTS
Total samples: 50
Correct predictions: 43
Accuracy: 86.00%

--- Confusion Matrix ---
True Positives (Allowed): 23
False Positives (Allowed): 2
False Negatives (Not Allowed): 5
True Negatives (Not Allowed): 20

--- Metrics for 'Allowed' Class ---
Precision: 92.00%
Recall: 82.14%
F1 Score: 86.79%

--- Metrics for 'Not Allowed' Class ---
Precision: 80.00%
Recall: 90.91%
F1 Score: 85.11%

Showing first 5 incorrect predictions:

------------------------------------------------------------
Incorrect Prediction #1
------------------------------------------------------------
Question: การตั้งชื่อสมาคมมีข้อกหนดหรือไม่...
Expected: allowed
Predicted: not allowed
Raw Response: NO

------------------------------------------------------------
Incorrect Prediction #2
------------------------------------------------------------
Question: หากได้รับความเสียหายจากการรับรองที่ไมไ่ด้มาตรฐานจะสามารถเรียกร้องค่าเสียหายได้หรือไม่...
Expected: not allowed
Predicted: allowed
Raw Response: Y

In [13]:
# ==========================================
# CELL 11: Run Large Sample Evaluation (200 samples)
# ==========================================
print("Running evaluation on 200 samples...")
metrics_sample, results_sample = run_evaluation(test_df, sample_size=200)
print_evaluation_results(metrics_sample, results_sample)

Running evaluation on 200 samples...


Evaluating: 100%|██████████| 200/200 [02:08<00:00,  1.55it/s]


EVALUATION RESULTS
Total samples: 200
Correct predictions: 177
Accuracy: 88.50%

--- Confusion Matrix ---
True Positives (Allowed): 92
False Positives (Allowed): 12
False Negatives (Not Allowed): 11
True Negatives (Not Allowed): 85

--- Metrics for 'Allowed' Class ---
Precision: 88.46%
Recall: 89.32%
F1 Score: 88.89%

--- Metrics for 'Not Allowed' Class ---
Precision: 88.54%
Recall: 87.63%
F1 Score: 88.08%

Showing first 5 incorrect predictions:

------------------------------------------------------------
Incorrect Prediction #1
------------------------------------------------------------
Question: การตั้งชื่อสมาคมมีข้อกหนดหรือไม่...
Expected: allowed
Predicted: not allowed
Raw Response: NO

------------------------------------------------------------
Incorrect Prediction #2
------------------------------------------------------------
Question: หากได้รับความเสียหายจากการรับรองที่ไมไ่ด้มาตรฐานจะสามารถเรียกร้องค่าเสียหายได้หรือไม่...
Expected: not allowed
Predicted: allowed
Raw Respons

In [ ]:
# ==========================================
# CELL 12: Save Results to CSV
# ==========================================
# Save the latest evaluation results
output_filename = 'evaluation_results_200.csv'
results_sample.to_csv(output_filename, index=False)
print(f"✓ Results saved to '{output_filename}'")

# Show summary statistics
print(f"\nSummary Statistics:")
print(results_sample['predicted_decision'].value_counts())

In [14]:
# ==========================================
# CELL 13: Run Full Dataset Evaluation (Optional)
# ==========================================
# WARNING: This may take a long time depending on dataset size
print(f"Running evaluation on FULL dataset ({len(test_df)} samples)...")
print("This may take a while...")
metrics_full, results_full = run_evaluation(test_df, sample_size=None)
print_evaluation_results(metrics_full, results_full)

# Save full results
#results_full.to_csv('evaluation_results_full.csv', index=False)
#print("✓ Full results saved to 'evaluation_results_full.csv'")

Running evaluation on FULL dataset (3729 samples)...
This may take a while...


Evaluating: 100%|██████████| 3729/3729 [39:00<00:00,  1.59it/s]


EVALUATION RESULTS
Total samples: 3729
Correct predictions: 3225
Accuracy: 86.48%

--- Confusion Matrix ---
True Positives (Allowed): 1421
False Positives (Allowed): 308
False Negatives (Not Allowed): 196
True Negatives (Not Allowed): 1804

--- Metrics for 'Allowed' Class ---
Precision: 82.19%
Recall: 87.88%
F1 Score: 84.94%

--- Metrics for 'Not Allowed' Class ---
Precision: 90.20%
Recall: 85.42%
F1 Score: 87.74%

Showing first 5 incorrect predictions:

------------------------------------------------------------
Incorrect Prediction #1
------------------------------------------------------------
Question: ถ้าผู้รับประกันภัยต้องคำพิพากษาให้เป็นคนล้มละลาย ผู้เอาประกันภัยต้องทำอย่างไร...
Expected: allowed
Predicted: not allowed
Raw Response: NO

------------------------------------------------------------
Incorrect Prediction #2
------------------------------------------------------------
Question: ถ้าผู้ให้หลักประกันจะนำทรัพย์สินที่ตนมีสิทธิจะได้มาในอนาคตตามสัญญามาใช้เป็นหลักประกันได้

In [ ]:
# ==========================================
# CELL 14: Analyze Results by Category (Optional)
# ==========================================
# Analyze false positives and false negatives
print("\n" + "="*60)
print("DETAILED ERROR ANALYSIS")
print("="*60)

false_positives = results_sample[
    (results_sample['predicted_decision'] == 'allowed') &
    (results_sample['expected_decision'] != 'allowed')
]
false_negatives = results_sample[
    (results_sample['predicted_decision'] != 'allowed') &
    (results_sample['expected_decision'] == 'allowed')
]

print(f"\nFalse Positives (predicted allowed, but not): {len(false_positives)}")
if len(false_positives) > 0:
    print("\nExamples:")
    for i, (idx, row) in enumerate(false_positives.head(3).iterrows(), 1):
        print(f"\n{i}. {row['question'][:100]}...")
        print(f"   Response: {row['raw_response']}")

print(f"\nFalse Negatives (predicted not allowed, but is): {len(false_negatives)}")
if len(false_negatives) > 0:
    print("\nExamples:")
    for i, (idx, row) in enumerate(false_negatives.head(3).iterrows(), 1):
        print(f"\n{i}. {row['question'][:100]}...")
        print(f"   Response: {row['raw_response']}")